In [6]:
import json
from collections import defaultdict

files = {
    'base': '/Users/novay/Applications/Project/Enterwind/lexindo/python/lexindo-llm/05_evaluation_set/ragas_eval_result_base.jsonl',
    'finetuned': '/Users/novay/Applications/Project/Enterwind/lexindo/python/lexindo-llm/05_evaluation_set/ragas_eval_result_finetuned.jsonl',
    'rag': '/Users/novay/Applications/Project/Enterwind/lexindo/python/lexindo-llm/05_evaluation_set/ragas_eval_finetuned_rag_batch.jsonl'
}

merged = defaultdict(dict)
merged

defaultdict(dict, {})

In [7]:
# Load base
with open(files['base'], 'r', encoding='utf-8') as f:
    for line in f:
        d = json.loads(line.strip())
        key = d['question']  # gunakan question sebagai key unik
        merged[key]['question'] = d['question']
        merged[key]['ground_truth'] = d.get('ground_truth', d.get('answer', ''))
        merged[key]['answer_base'] = d.get('answer_base', '')
        merged[key]['contexts'] = d.get('contexts', [])

In [8]:
# Load finetuned no-RAG (override/isi kolom baru)
with open(files['finetuned'], 'r', encoding='utf-8') as f:
    for line in f:
        d = json.loads(line.strip())
        key = d['question']
        if key in merged:
            merged[key]['answer_finetuned_no_rag'] = d.get('answer', '')
        else:
            # kalau ada yang tidak match, tambah baru
            merged[key] = {
                'question': d['question'],
                'ground_truth': d.get('ground_truth', ''),
                'answer_finetuned_no_rag': d.get('answer', '')
            }

In [9]:
# Load +RAG (override contexts & answer_rag)
with open(files['rag'], 'r', encoding='utf-8') as f:
    for line in f:
        d = json.loads(line.strip())
        key = d['question']
        if key in merged:
            merged[key]['answer_finetuned_rag'] = d.get('answer_finetuned_rag', '')
            merged[key]['contexts'] = d.get('contexts', merged[key]['contexts'])
        else:
            merged[key] = {
                'question': d['question'],
                'ground_truth': d.get('ground_truth', ''),
                'answer_finetuned_rag': d.get('answer_finetuned_rag', ''),
                'contexts': d.get('contexts', [])
            }

In [11]:
# Simpan merged
output_merged = '/Users/novay/Applications/Project/Enterwind/lexindo/python/lexindo-llm/07_experiments/ragas_eval_merged.jsonl'
with open(output_merged, 'w', encoding='utf-8') as f:
    for entry in merged.values():
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

print(f"Merged selesai: {len(merged)} entries → {output_merged}")

Merged selesai: 1109 entries → /Users/novay/Applications/Project/Enterwind/lexindo/python/lexindo-llm/07_experiments/ragas_eval_merged.jsonl
